# Experiment 9: Implement ResNet for Detecting Objects

**Aim:** Build a Residual Network (ResNet) with skip connections and use it as the backbone of a simple **sliding-window object detector**.

**Idea:**
1. Implement a ResNet (ResNet-20 style for 32x32 images) with identity / projection shortcut residual blocks.
2. Train it as an 11-way patch classifier: the 10 CIFAR-10 object classes + a **background** class.
3. Build synthetic multi-object scenes (CIFAR-10 objects pasted on 96x96 backgrounds, with known bounding boxes).
4. Detect objects by sliding a 32x32 window over a scene, classifying each window with the ResNet, keeping confident non-background windows and applying **Non-Maximum Suppression (NMS)**.
5. Evaluate the detector with IoU-based precision / recall.

**Dataset:** CIFAR-10 (60,000 32x32 colour images, 10 classes).

> Note: Full detectors such as Faster R-CNN / SSD / YOLO use a ResNet as a *feature extractor* too, but add region-proposal or box-regression heads. The sliding-window approach here keeps the focus on the ResNet while still producing real bounding boxes.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 0
rng = np.random.default_rng(SEED); tf.random.set_seed(SEED)
EPOCHS = 12
N_TRAIN_SCENES = 6000
CLASSES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck","background"]
BG = 10
print("TensorFlow", tf.__version__)

## 1. The ResNet: residual blocks
A residual block computes `y = F(x) + shortcut(x)`. When the spatial size / channels change, the shortcut is a 1x1 strided convolution (projection); otherwise it is the identity. Skip connections let gradients flow directly through the network, which makes deep networks trainable.

In [ ]:
def residual_block(x, filters, stride=1):
    shortcut = x
    y = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)
    y = layers.BatchNormalization()(y); y = layers.ReLU()(y)
    y = layers.Conv2D(filters, 3, padding="same", use_bias=False)(y)
    y = layers.BatchNormalization()(y)
    if stride != 1 or x.shape[-1] != filters:                       # projection shortcut
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False)(x)
        shortcut = layers.BatchNormalization()(shortcut)
    y = layers.Add()([y, shortcut])
    return layers.ReLU()(y)

def build_resnet(n=3, num_classes=len(CLASSES), input_shape=(32, 32, 3)):
    """ResNet-(6n+2): n=3 -> ResNet-20."""
    inp = layers.Input(input_shape)
    x = layers.Conv2D(16, 3, padding="same", use_bias=False)(inp)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    for filters, stride in [(16, 1), (32, 2), (64, 2)]:
        for b in range(n):
            x = residual_block(x, filters, stride if b == 0 else 1)
    x = layers.GlobalAveragePooling2D()(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inp, out, name=f"ResNet{6*n+2}")

model = build_resnet()
model.summary(line_length=100, expand_nested=False)

## 2. Dataset: synthetic detection scenes built from CIFAR-10
Each scene is a 96x96 smooth random-colour background (gradient + noise) with 1-3 CIFAR-10 objects pasted at random locations (non-overlapping). Ground-truth boxes are stored as `(x1, y1, x2, y2, class)`.

In [ ]:
(x_tr, y_tr), (x_te, y_te) = keras.datasets.cifar10.load_data()
y_tr, y_te = y_tr.ravel(), y_te.ravel()
SCENE, WIN = 96, 32

def random_background(r):
    c0, c1 = r.uniform(0, 255, 3), r.uniform(0, 255, 3)
    t = np.linspace(0, 1, SCENE)[None, :, None] if r.random() < .5 else np.linspace(0, 1, SCENE)[:, None, None]
    bg = c0 + (c1 - c0) * t * np.ones((SCENE, SCENE, 1))
    bg = bg + r.normal(0, 8, bg.shape)
    return np.clip(bg, 0, 255)

def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union

def make_scene(imgs, labels, r):
    canvas, boxes = random_background(r), []
    for _ in range(r.integers(1, 4)):
        for _try in range(30):
            x, y = r.integers(0, SCENE - WIN + 1, 2)
            box = (x, y, x + WIN, y + WIN)
            if all(iou(box, b[:4]) == 0 for b in boxes):
                k = r.integers(len(imgs))
                canvas[y:y+WIN, x:x+WIN] = imgs[k]
                boxes.append((*box, int(labels[k]))); break
    return canvas.astype("float32") / 255.0, boxes

def crop(scene, x, y):
    return scene[y:y+WIN, x:x+WIN]

def build_patch_dataset(n_scenes, imgs, labels, r):
    X, Y = [], []
    for _ in range(n_scenes):
        scene, boxes = make_scene(imgs, labels, r)
        for (x1, y1, x2, y2, c) in boxes:                       # object patches (jittered)
            for _ in range(2):
                dx, dy = r.integers(-3, 4, 2)
                nx, ny = np.clip(x1+dx, 0, SCENE-WIN), np.clip(y1+dy, 0, SCENE-WIN)
                if iou((nx, ny, nx+WIN, ny+WIN), (x1, y1, x2, y2)) >= 0.6:
                    X.append(crop(scene, nx, ny)); Y.append(c)
        got = 0
        while got < 5:                                          # background patches (IoU < 0.1 with all objects)
            x, y = r.integers(0, SCENE - WIN + 1, 2)
            if all(iou((x, y, x+WIN, y+WIN), b[:4]) < 0.1 for b in boxes):
                X.append(crop(scene, x, y)); Y.append(BG); got += 1
    return np.array(X, "float32"), np.array(Y)

X_train, Y_train = build_patch_dataset(N_TRAIN_SCENES, x_tr, y_tr, rng)
X_val,   Y_val   = build_patch_dataset(800, x_te, y_te, np.random.default_rng(1))
print("Patches:", X_train.shape, "  class counts:", np.bincount(Y_train))

fig, ax = plt.subplots(1, 4, figsize=(11, 3))
for a in ax:
    s, bx = make_scene(x_te, y_te, rng); a.imshow(s); a.axis("off")
    for (x1,y1,x2,y2,c) in bx:
        a.add_patch(patches.Rectangle((x1,y1), WIN, WIN, fill=False, ec="lime", lw=2)); a.text(x1, y1-2, CLASSES[c], color="lime", fontsize=8)
plt.suptitle("Example synthetic scenes with ground-truth boxes"); plt.show()

## 3. Train the ResNet patch classifier

In [ ]:
aug = keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomTranslation(0.05, 0.05)])
inp = layers.Input((32, 32, 3)); out = model(aug(inp)); train_model = keras.Model(inp, out)
train_model.compile(optimizer=keras.optimizers.Adam(2e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cb = [keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, verbose=1)]
hist = train_model.fit(X_train, Y_train, validation_data=(X_val, Y_val), epochs=EPOCHS, batch_size=128, callbacks=cb, verbose=2)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="val"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(hist.history["accuracy"], label="train"); ax[1].plot(hist.history["val_accuracy"], label="val"); ax[1].set_title("Accuracy"); ax[1].legend()
plt.show()

## 4. Sliding-window detection + Non-Maximum Suppression
All 32x32 windows (stride 4) of a scene are classified in one batch. Windows whose best non-background class probability exceeds a threshold become candidate boxes; NMS then removes duplicates, keeping the highest-scoring box among heavily overlapping ones.

In [ ]:
def nms(dets, iou_thr=0.3):
    dets = sorted(dets, key=lambda d: -d[5]); keep = []
    for d in dets:
        if all(iou(d[:4], k[:4]) < iou_thr for k in keep): keep.append(d)
    return keep

def detect(scene, stride=4, score_thr=0.9, iou_thr=0.3):
    coords = [(x, y) for y in range(0, SCENE-WIN+1, stride) for x in range(0, SCENE-WIN+1, stride)]
    batch = np.stack([crop(scene, x, y) for x, y in coords])
    probs = model.predict(batch, verbose=0)
    dets = []
    for (x, y), p in zip(coords, probs):
        c = int(p[:BG].argmax())
        if p[BG] < 0.5 and p[c] >= score_thr:
            dets.append((x, y, x+WIN, y+WIN, c, float(p[c])))
    return nms(dets, iou_thr)

test_rng = np.random.default_rng(123)
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for a in axes.ravel():
    s, gt = make_scene(x_te, y_te, test_rng); dets = detect(s)
    a.imshow(s); a.axis("off")
    for (x1,y1,x2,y2,c) in gt:
        a.add_patch(patches.Rectangle((x1,y1), WIN, WIN, fill=False, ec="lime", lw=1.5, ls="--"))
    for (x1,y1,x2,y2,c,sc) in dets:
        a.add_patch(patches.Rectangle((x1,y1), WIN, WIN, fill=False, ec="red", lw=2))
        a.text(x1, y2+8, f"{CLASSES[c]} {sc:.2f}", color="white", fontsize=8, backgroundcolor="red")
plt.suptitle("Green dashed = ground truth, Red = ResNet detections"); plt.tight_layout(); plt.show()

## 5. Quantitative evaluation (IoU >= 0.5)
A detection is a **true positive** if it matches an unused ground-truth box with IoU >= 0.5 **and** the predicted class is correct.

In [ ]:
def evaluate(n_scenes=200, seed=999, **kw):
    r = np.random.default_rng(seed); TP = FP = FN = 0
    for _ in range(n_scenes):
        s, gt = make_scene(x_te, y_te, r); dets = detect(s, **kw); used = set()
        for d in dets:
            hit = None
            for j, g in enumerate(gt):
                if j not in used and iou(d[:4], g[:4]) >= 0.5 and d[4] == g[4]:
                    hit = j; break
            if hit is None: FP += 1
            else: TP += 1; used.add(hit)
        FN += len(gt) - len(used)
    prec, rec = TP / max(TP+FP, 1), TP / max(TP+FN, 1)
    return prec, rec, 2*prec*rec/max(prec+rec, 1e-9)

for thr in (0.7, 0.9, 0.97):
    p, r_, f = evaluate(score_thr=thr)
    print(f"score_thr={thr}:  precision={p:.3f}  recall={r_:.3f}  F1={f:.3f}")

## Conclusion
* A ResNet-20 with residual (skip) connections was implemented from scratch and trained to recognise the 10 CIFAR-10 object classes plus background.
* Using it as the classifier in a sliding-window + NMS pipeline produced bounding boxes with class labels on multi-object scenes. The precision/recall trade-off is controlled by the score threshold.
* Production detectors (Faster R-CNN, RetinaNet, SSD) replace the exhaustive sliding window with learned region proposals/box regression but reuse exactly this kind of ResNet backbone.